## Notebook10

### Setup

Run all of the following before starting the notebook.

In [2]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [3]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [4]:
us_pop = (
    pl.read_csv(ub + "data/us_city_population.csv")
    .select(c.city, c.state, c.year, c.population)
    .filter(c.population.null_count().over(c.city) <= 1)
)

### Questions

This week's data is the population of 289 American cities at every census from
1790 to 2010, which makes 6,647 rows. Each row is one city in one census year.
The `population` column is in thousands, so New York City in 2010 is 8175.133 and
not 8,175,133. A city that did not exist yet has a population of 0 rather than a
missing value, which matters more than it sounds like it should.

The cities are roughly the largest ones in the country today, traced backward
through time. That means the table is not a sample of American cities and it is
certainly not all of them: 46 states appear, some of them with a single city.
Keep that in mind whenever you compute a total, because the total is over the
cities that happen to be here.

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Start with the tools you already have. For the 2010 census only, compute each
state's total city population and the number of cities it contributes, and sort
the states from largest to smallest.

In [5]:
(
    us_pop
    .filter(c.year == 2010)
    .group_by(c.state)
    .agg(
        pop_total = c.population.sum(),
        n = pl.len()
    )
    .sort(c.pop_total, descending=True)
)

state,pop_total,n
str,f64,u32
"""CA""",18231.904,71
"""TX""",10719.818,29
"""NY""",9086.01,6
"""IL""",3764.177,9
"""FL""",3666.925,16
…,…,…
"""RI""",178.042,1
"""MS""",173.514,1
"""SD""",153.888,1


California leads with 71 cities and 18.2 million people in them. Now suppose you
want each city's share of its own state's total. The share needs two numbers on
the same row: the city's population, which you have, and the state total, which is
sitting in a different table with one row per state. Aggregation put it there and
aggregation cannot get it back.

2. `.over()` can. Take the 2010 census, write each state's total onto every city in
that state, and compute the share. Sort by share, largest first.

In [6]:
(
    us_pop
    .filter(c.year == 2010)
    .with_columns(
        share = c.population / c.population.sum().over(c.state) * 100
    )
    .sort(c.share, descending=True)
)

city,state,year,population,share
str,str,i64,f64,f64
"""Anchorage, AK""","""AK""",2010,291.826,100.0
"""Little Rock, AR""","""AR""",2010,193.524,100.0
"""Washington, DC""","""DC""",2010,601.723,100.0
"""Honolulu, HI""","""HI""",2010,387.17,100.0
"""Boise, ID""","""ID""",2010,205.671,100.0
…,…,…,…,…
"""Mission Viejo, CA""","""CA""",2010,93.305,0.511768
"""Vacaville, CA""","""CA""",2010,92.428,0.506957
"""Carson, CA""","""CA""",2010,91.714,0.503041


The top of the result is a column of 100.0. Anchorage is 100 percent of Alaska,
Little Rock is 100 percent of Arkansas, and so on down a list of states that
contribute exactly one city to the table. The share is a share of the cities in
*this table*, and question 1's `n` column showed that for those states the table
holds one city.

3. So look at the states that have enough cities for the share to mean anything.
Keep only the states with at least five cities in the table, then compute the same
share. The count is itself a window: `pl.len().over(c.state)` gives every row the
number of rows in its state, and a window expression can go straight inside a
`.filter()`.

In [7]:
(
    us_pop
    .filter(c.year == 2010)
    .filter(pl.len().over(c.state) >= 5)
    .with_columns(
        share = c.population / c.population.sum().over(c.state) * 100
    )
    .sort(c.share, descending=True)
)

city,state,year,population,share
str,str,i64,f64,f64
"""New York City, NY""","""NY""",2010,8175.133,89.974951
"""Chicago, IL""","""IL""",2010,2695.598,71.611882
"""Indianapolis, IN""","""IN""",2010,820.445,59.75447
"""Seattle, WA""","""WA""",2010,608.66,43.378237
"""Detroit, MI""","""MI""",2010,713.777,40.20314
…,…,…,…,…
"""Mission Viejo, CA""","""CA""",2010,93.305,0.511768
"""Vacaville, CA""","""CA""",2010,92.428,0.506957
"""Carson, CA""","""CA""",2010,91.714,0.503041


New York City is 90 percent of New York's urban population and Chicago is 72
percent of Illinois's. Nothing is added to the table in the filter and nothing is
removed from it in the `with_columns`. The same expression does different work
depending on which context you drop it into.

4. Here is a comparison that needs the rows preserved. Keep the 2010 cities whose
population is above the mean city population of their own state.

In [8]:
(
    us_pop
    .filter(c.year == 2010)
    .filter(c.population > c.population.mean().over(c.state))
)

city,state,year,population
str,str,i64,f64
"""Birmingham, AL""","""AL""",2010,212.237
"""Montgomery, AL""","""AL""",2010,205.764
"""Mesa, AZ""","""AZ""",2010,439.041
"""Phoenix, AZ""","""AZ""",2010,1445.632
"""Tucson, AZ""","""AZ""",2010,520.116
…,…,…,…
"""Norfolk, VA""","""VA""",2010,242.803
"""Richmond, VA""","""VA""",2010,204.214
"""Virginia Beach, VA""","""VA""",2010,437.994


Seventy-six cities of 289, which is barely a quarter of them. That is what a mean
does when one city in the state is enormous: Los Angeles drags California's average
up past most of the state, and the average stops describing a typical city. Swap
`.mean()` for `.median()` and 132 cities come back instead, much nearer to half, as
you would expect from a median.

5. Every window so far has grouped by state and stayed inside one year. Turn it
around: group by *year* and look across the states. Give each city its share of all
289 cities' population in its census year, then follow New York City down the
column.

In [9]:
(
    us_pop
    .with_columns(
        share = c.population / c.population.sum().over(c.year) * 100
    )
    .filter(c.city == "New York City, NY")
)

city,state,year,population,share
str,str,i64,f64,f64
"""New York City, NY""","""NY""",1790,33.131,23.488667
"""New York City, NY""","""NY""",1800,60.515,26.212289
"""New York City, NY""","""NY""",1810,96.373,25.634526
"""New York City, NY""","""NY""",1820,123.706,25.336662
"""New York City, NY""","""NY""",1830,202.589,25.730554
…,…,…,…,…
"""New York City, NY""","""NY""",1970,7895.563,12.405747
"""New York City, NY""","""NY""",1980,7071.639,10.832916
"""New York City, NY""","""NY""",1990,7322.564,10.262816


New York in 2010 is nearly twice the size it was in 1910, yet its share peaked at
19.2 percent of these cities in 1910 and has fallen to 9.6 percent since. Both
statements come from the same two columns. The city got bigger; the country's other
cities got bigger faster.

6. Now for the ordered windows. Sort the table by city and year and compute how
much each city gained since the previous census by subtracting the shifted column,
with `.over(c.city)` so that the shift restarts at each city. Then drop the nulls,
sort by the change, and see which decade of which city added the most people.

In [10]:
(
    us_pop
    .filter(c.year == 2010)
    .group_by(c.state)
    .agg(
        pop_total = c.population.sum(),
        n = pl.len()
    )
    .sort(c.pop_total, descending=True)
)

state,pop_total,n
str,f64,u32
"""CA""",18231.904,71
"""TX""",10719.818,29
"""NY""",9086.01,6
"""IL""",3764.177,9
"""FL""",3666.925,16
…,…,…
"""RI""",178.042,1
"""MS""",173.514,1
"""SD""",153.888,1


The first census of every city now has a `null` change, because there is no row
above it to subtract. That is why the `.drop_nulls()` is there: without it Polars
sorts those nulls to the top of a descending sort and you read a screen of blanks.
What is left is New York City adding 1.9 million people
between 1890 and 1900, then 1.3 million more in the next ten years, with Chicago,
Los Angeles, and Detroit arriving in the 1920s. Nine of the ten largest gains happen
between 1880 and 1930, when the industrial cities were growing. The tenth is New
York again, in the 1990s.

7. Sort the other way and you get the second half of that story. Keep only the 2010
rows with a negative change, so the question becomes: which cities lost people
between 2000 and 2010?

In [11]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(
        change = c.population - c.population.shift(1).over(c.city)
    )
    .filter(c.year == 2010, c.change < 0)
    .sort(c.change)
)

city,state,year,population,change
str,str,i64,f64,f64
"""Detroit, MI""","""MI""",2010,713.777,-237.493
"""Chicago, IL""","""IL""",2010,2695.598,-200.418
"""New Orleans, LA""","""LA""",2010,343.829,-140.845
"""Cleveland, OH""","""OH""",2010,396.815,-81.588
"""Cincinnati, OH""","""OH""",2010,296.943,-34.342
…,…,…,…,…
"""Miami Beach, FL""","""FL""",2010,87.779,-0.154
"""Nashua, NH""","""NH""",2010,86.494,-0.111
"""Yonkers, NY""","""NY""",2010,195.976,-0.11


Sixty-one cities shrank in that decade. Detroit lost 237,000 people and Chicago
200,000, part of a long decline. New Orleans lost 141,000 to a single hurricane in
August 2005, which falls between two census counts taken five years on either side
of it. A decade is a coarse instrument, and the column does not tell you which of
these two kinds of loss you are looking at.

8. Absolute change favors big cities, so try growth as a percentage instead. In
the first block, use the same shifted column as a denominator, drop the nulls, and
sort with the fastest growth first. Look at what comes back. In the second block,
save the shifted column as `prev` first, keep only the rows where `prev > 0`, and
then compute the growth and sort again.

In [13]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(
        growth = (c.population - c.population.shift(1).over(c.city))
            / c.population.shift(1).over(c.city) * 100
    )
    .drop_nulls()
    .sort(c.growth, descending=True)
)

(
    us_pop
    .sort(c.city, c.year)
    .with_columns(prev = c.population.shift(1).over(c.city))
    .filter(c.prev > 0)
    .with_columns(growth = (c.population - c.prev) / c.prev * 100)
    .sort(c.growth, descending=True)
)

city,state,year,population,prev,growth
str,str,i64,f64,f64,f64
"""Warren, MI""","""MI""",1960,89.246,0.7,12649.428571
"""Coral Springs, FL""","""FL""",1980,37.349,1.5,2389.933333
"""Garden Grove, CA""","""CA""",1960,84.238,3.762,2139.181287
"""Virginia Beach, VA""","""VA""",1970,172.106,8.091,2027.128909
"""Irving, TX""","""TX""",1960,45.985,2.621,1654.483022
…,…,…,…,…,…
"""Springfield, MO""","""MO""",1860,0.0,0.4,-100.0
"""Sterling Heights, MI""","""MI""",1950,0.0,0.4,-100.0
"""Tempe, AZ""","""AZ""",1910,0.0,0.8,-100.0


**Answer**:


The second block drops only the rows with a zero denominator, and every remaining
row still compares itself to the census immediately before it. At the top is Warren,
Michigan: 700 people in 1950, 89,246 in 1960, a 12,649 percent decade. That one is
real, and it is what a Detroit suburb looked like when Detroit was still growing.
Percent growth rewards a small denominator, so it hands you villages that became
suburbs and it will never once mention New York, while question 6 hands you nothing
but New York. Both are correct. Choose the one that matches the question you were
asked. Notice too that the filter comes after the window: filtering out the zeros
first would change what "the row above" means, and the shift would silently compare
censuses decades apart.

9. One city in this table has an actual hole in it. Portland, Oregon has no
population recorded for 1850. Sort by city and year, then use `.forward_fill()` and
`.backward_fill()`, both scoped with `.over(c.city)`, to fill the gap from above and
from below, and look at Portland from 1830 to 1870.

In [14]:
(
    us_pop
    .sort(c.city, c.year)
    .with_columns(
        pop_ffill = c.population.forward_fill().over(c.city),
        pop_bfill = c.population.backward_fill().over(c.city)
    )
    .filter(c.city == "Portland, OR", c.year.is_between(1830, 1870))
)

city,state,year,population,pop_ffill,pop_bfill
str,str,i64,f64,f64,f64
"""Portland, OR""","""OR""",1830,0.0,0.0,0.0
"""Portland, OR""","""OR""",1840,0.0,0.0,0.0
"""Portland, OR""","""OR""",1850,null,0.0,2.874
"""Portland, OR""","""OR""",1860,2.874,2.874,2.874
"""Portland, OR""","""OR""",1870,8.293,8.293,8.293


Forward fill copies 1840 down and says Portland had nobody in it. Backward fill
copies 1860 up and says it had 2,874 people. The census counted 821. Both fills are
guesses, wrong in opposite directions, with the true value between them. That is
generally the most you can expect from a fill. Note also that the `.over()`
changed nothing here, since the row above Portland's gap happens to be Portland. It
would have changed everything if the gap had landed on a city's first row, and you do
not get to know that in advance.